# 07 - Grouped start, stop and persistence

Read this notebook and paste each Bash block into an Ubuntu terminal, in order. Keep the same terminal for the commands in this notebook. These are shell instructions; do not run them in a Python kernel.

## Open the project

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



`all` means the four initial services: S3, PostgreSQL, Polaris and MySQL. It does not start the later Kafka, Spark, Trino, Airflow or monitoring sessions. `cluster.sh` always uses context/namespace `bigdata` and processes services in dependency order. Stopping retains PVCs and example data; deleting a cluster is different.

## Start the initial group

```bash
bash cluster.sh start all
```

Order: S3 - PostgreSQL - Polaris - MySQL. Dependencies are awaited before proceeding.

## Inspect the initial group

```bash
bash cluster.sh status all
kubectl get pvc,pv
```



## Produce a real SQL log record

```bash
MONITORING_MARKER="workspace-monitoring-$(date +%s)"
printf '%s
' "$MONITORING_MARKER" > .state/monitoring.marker
kubectl exec postgres-0 -- sh -ec 'export PGPASSWORD="$POSTGRES_PASSWORD"; exec psql -h postgres -U "$POSTGRES_USER" -d "$POSTGRES_DB" -v ON_ERROR_STOP=1 -c "SET log_statement='\''all'\'';" -c "SELECT '\''$1'\'' AS monitoring_example;"' -- "$MONITORING_MARKER"
python3 validations/check-observability.py --components postgres --marker "$MONITORING_MARKER"
```

This uses session-only SQL statement logging to produce an actual PostgreSQL daemon log. It does not change global database settings. The same record is queried again after restarting monitoring below.


## Stop the initial group

```bash
bash cluster.sh stop all
```

Stops in reverse order, retains all PVCs and the completed bootstrap Job.

## Stop the Minikube node

```bash
minikube stop -p bigdata
```

This keeps the profile and all its PVC data.

## Resume the Minikube node

```bash
source versions.env
minikube start -p bigdata --driver=docker --container-runtime=containerd --kubernetes-version="$LAB_K8S_VERSION" --cpus="$LAB_CPUS" --memory="$LAB_MEMORY_MB"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
kubectl wait --for=condition=Ready node --all --timeout=180s
```



## Resume the initial group

```bash
bash cluster.sh start all
```



## Wait for monitoring after the node restart

```bash
for component in loki alloy prometheus grafana; do
  kubectl rollout status "statefulset/$component" --timeout=600s
done
```


## Check the retained SQL log record

```bash
MONITORING_MARKER=$(cat .state/monitoring.marker)
python3 validations/check-observability.py --components postgres --marker "$MONITORING_MARKER"
```

The original database query is not repeated here. Finding its same marker after the Minikube node stop/resume demonstrates retained Loki data.


## Check PostgreSQL persisted rows

```bash
kubectl exec postgres-0 -- sh -ec 'export PGPASSWORD="$POSTGRES_PASSWORD"; psql -h postgres -U "$POSTGRES_USER" -d test_database -v ON_ERROR_STOP=1 -c "SELECT * FROM test_table;"'
```



## Check MySQL persisted rows

```bash
kubectl exec mysql-0 -- sh -ec 'MYSQL_PWD="$MYSQL_PASSWORD" exec mysql --protocol=TCP -h mysql -u "$MYSQL_USER" test_database -e "SELECT * FROM test_table;"'
```



## Forward persisted storage APIs

```bash
if [[ -f .state/s3-forward.pid ]]; then
  FORWARD_PID=$(cat .state/s3-forward.pid)
  if [[ "$(ps -p "$FORWARD_PID" -o args= 2>/dev/null || true)" == 'kubectl port-forward --address=127.0.0.1 service/seaweedfs 19001:8333' ]]; then kill "$FORWARD_PID"; fi
fi
kubectl port-forward --address=127.0.0.1 service/seaweedfs 19001:8333 > .state/s3-forward.log 2>&1 &
echo $! > .state/s3-forward.pid
if [[ -f .state/polaris-forward.pid ]]; then
  FORWARD_PID=$(cat .state/polaris-forward.pid)
  if [[ "$(ps -p "$FORWARD_PID" -o args= 2>/dev/null || true)" == 'kubectl port-forward --address=127.0.0.1 service/polaris 18181:8181 18182:8182' ]]; then kill "$FORWARD_PID"; fi
fi
kubectl port-forward --address=127.0.0.1 service/polaris 18181:8181 18182:8182 > .state/polaris-forward.log 2>&1 &
echo $! > .state/polaris-forward.pid
for attempt in $(seq 1 60); do
  if curl -fsS http://127.0.0.1:18182/q/health/ready >/dev/null 2>&1; then break; fi
  sleep 1
done
curl -fsS http://127.0.0.1:18182/q/health/ready
```



## Read the existing S3 object

```bash
S3_ACCESS=$(kubectl get secret lab-credentials -o jsonpath='{.data.s3-access-key}' | base64 -d)
S3_SECRET=$(kubectl get secret lab-credentials -o jsonpath='{.data.s3-secret-key}' | base64 -d)
curl -fsS --aws-sigv4 aws:amz:us-east-1:s3 --user "$S3_ACCESS:$S3_SECRET" http://127.0.0.1:19001/test-bucket/test-object.txt > .state/persisted-object.txt
cmp .state/test-object.txt .state/persisted-object.txt
cat .state/persisted-object.txt
```



## Authenticate after the catalog restart

```bash
POLARIS_URL=http://127.0.0.1:18181
CLIENT_ID=$(kubectl get secret lab-credentials -o jsonpath='{.data.polaris-client-id}' | base64 -d)
CLIENT_SECRET=$(kubectl get secret lab-credentials -o jsonpath='{.data.polaris-client-secret}' | base64 -d)
TOKEN=$(curl -fsS "$POLARIS_URL/api/catalog/v1/oauth/tokens" -H 'Polaris-Realm: POLARIS' --data-urlencode grant_type=client_credentials --data-urlencode "client_id=$CLIENT_ID" --data-urlencode "client_secret=$CLIENT_SECRET" --data-urlencode scope=PRINCIPAL_ROLE:ALL | python3 -c 'import json,sys; print(json.load(sys.stdin)["access_token"])')
api() { curl -fsS -H "Authorization: Bearer $TOKEN" -H 'Polaris-Realm: POLARIS' -H 'Content-Type: application/json' "$@"; }
```



## Compare persisted Iceberg metadata

```bash
api "$POLARIS_URL/api/catalog/v1/test_catalog/namespaces/test_namespace/tables/test_table" > .state/persisted-table.json
python3 - <<'PY'
import json
before = json.load(open('.state/iceberg-table.json'))
after = json.load(open('.state/persisted-table.json'))
assert before['metadata']['table-uuid'] == after['metadata']['table-uuid']
assert before['metadata-location'] == after['metadata-location']
print('PASS: same Iceberg table UUID and metadata location after service and node restart.')
PY
kubectl get job polaris-bootstrap
```



## Close validation forwards

```bash
kill "$(cat .state/s3-forward.pid)" "$(cat .state/polaris-forward.pid)"
rm .state/s3-forward.pid .state/polaris-forward.pid
```



Other group examples:

- `bash cluster.sh start s3 postgres` starts only those stores.
- `bash cluster.sh start polaris` also starts its S3/PostgreSQL dependencies.
- `bash cluster.sh stop mysql polaris` leaves S3 and PostgreSQL running.
- `bash cluster.sh stop postgres` also stops Polaris first.

To stop the entire Minikube node while retaining data, run `minikube stop -p bigdata`. Resume with the same `minikube start` command from notebook 01 and `bash cluster.sh start all`.

To inspect a failing service use `kubectl describe pod <pod-name>` and `kubectl logs <pod-name>`. Use Kubernetes exec for processes inside pods. Docker exec targets the node container and is reserved for node diagnostics; it is not the database or S3 client command.

## Optional destructive reset

Skip this block when continuing to the next setup notebook. Run it only when intentionally discarding every PVC and database in this profile:

```bash
minikube delete -p bigdata
```

Repeat notebooks 01->06 to recreate an empty setup. Do not use this command for an ordinary stop.